# 01 - Data Profiling

Reproduces the Phase 1 profiling findings directly from the source Excel workbook. This notebook is the auditable record behind every number used in the pipeline design: anyone can re-run it against the same workbook and recover the same figures.

**Method.** Every sheet is read as raw text (no type coercion) so the profiler sees exactly what BigQuery's raw layer sees. Cleaning and casting belong in dbt staging, not here.

**Sections**
1. Shape and dtypes
2. Duplicate analysis (PK duplicates, attribute conflicts)
3. Null and invalid values
4. Broken foreign keys
5. Currency distribution
6. Late-arrival lag
7. Future-dated records
8. Inactive status
9. Branch name vs city inconsistency

A final comparison table checks each computed metric against the Phase 1 measurement.

In [ ]:
# --- Parameters ---
# Override EXCEL_PATH if the workbook lives elsewhere. The default searches a
# few common locations relative to where the notebook is launched from.
import os

WORKBOOK_NAME = 'Data Engineer Analyst - Study Case.xlsx'
_CANDIDATES = [
    WORKBOOK_NAME,
    os.path.join('..', WORKBOOK_NAME),
    os.path.join('..', '..', WORKBOOK_NAME),
]
EXCEL_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), _CANDIDATES[-1])

# Reference date for the future-dated check. None = today. Phase 1 was measured
# on 2026-06-03; set REFERENCE_DATE = '2026-06-03' to reproduce those exact
# counts (transaction_date 64, updated_at 87).
REFERENCE_DATE = None

print('Workbook:', EXCEL_PATH, '| exists:', os.path.exists(EXCEL_PATH))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

# Read every column as text and keep blanks as empty strings, so nothing is
# silently coerced (matches the all-STRING raw layer).
_read = dict(dtype=str, keep_default_na=False)
customers = pd.read_excel(EXCEL_PATH, sheet_name='customers', **_read)
transactions = pd.read_excel(EXCEL_PATH, sheet_name='transactions', **_read)
branches = pd.read_excel(EXCEL_PATH, sheet_name='branches', **_read)

# Collect computed metrics here for the final comparison table.
metrics = {}
print('Loaded:', len(customers), 'customers |', len(transactions), 'transactions |', len(branches), 'branches')

## Section 1: Shape and dtypes

Expected: customers 7,100 rows / 12 cols, transactions 12,300 rows / 14 cols, branches 30 rows / 8 cols. All columns read as `object` (raw text) by design.

In [ ]:
for name, df in [('customers', customers), ('transactions', transactions), ('branches', branches)]:
    print(f'\n=== {name}: {df.shape[0]} rows x {df.shape[1]} cols ===')
    print('columns:', list(df.columns))

metrics['customers_rows'] = len(customers)
metrics['transactions_rows'] = len(transactions)
metrics['branches_rows'] = len(branches)

## Section 2: Duplicate analysis

Primary-key duplicates and, for customers, which attributes actually conflict inside a duplicate pair.

Expected:
- customers: 100 duplicate `customer_id` values (200 rows). Conflicts on email/phone/birth_date (100/100), occupation (82), income_band (68), kyc_status (66). Never conflict: customer_name, city, customer_status, customer_segment.
- transactions: 300 duplicate `transaction_id` (12,000 unique). Every pair differs on at least `updated_at`.
- branches: no duplicates.

In [ ]:
# --- customers PK duplicates ---
cust_dup_mask = customers['customer_id'].duplicated(keep=False)
cust_dups = customers[cust_dup_mask]
n_cust_dup_ids = cust_dups['customer_id'].nunique()
n_cust_dup_rows = len(cust_dups)
print(f'customers: {n_cust_dup_ids} duplicate customer_id values across {n_cust_dup_rows} rows')

# Attribute conflict: how many duplicate-id groups carry >1 distinct value per column.
conflict_cols = ['email', 'phone_number', 'birth_date', 'occupation', 'income_band',
                 'kyc_status', 'customer_name', 'city', 'customer_status', 'customer_segment']
grp = cust_dups.groupby('customer_id')
conflicts = {c: int((grp[c].nunique() > 1).sum()) for c in conflict_cols}
conflict_tbl = pd.DataFrame.from_dict(conflicts, orient='index', columns=['groups_in_conflict'])
conflict_tbl = conflict_tbl.sort_values('groups_in_conflict', ascending=False)
display(conflict_tbl)

metrics['cust_dup_ids'] = n_cust_dup_ids
metrics['cust_dup_rows'] = n_cust_dup_rows
metrics['conflict_email'] = conflicts['email']
metrics['conflict_phone'] = conflicts['phone_number']
metrics['conflict_birth_date'] = conflicts['birth_date']
metrics['conflict_occupation'] = conflicts['occupation']
metrics['conflict_income_band'] = conflicts['income_band']
metrics['conflict_kyc_status'] = conflicts['kyc_status']

In [ ]:
# --- transactions PK duplicates ---
txn_dup_mask = transactions['transaction_id'].duplicated(keep=False)
txn_dups = transactions[txn_dup_mask]
n_txn_dup_ids = txn_dups['transaction_id'].nunique()
n_txn_unique = transactions['transaction_id'].nunique()

# Confirm every duplicate pair differs on updated_at.
tg = txn_dups.groupby('transaction_id')
pairs_differ_updated_at = int((tg['updated_at'].nunique() > 1).sum())
print(f'transactions: {n_txn_dup_ids} duplicate transaction_id values, {n_txn_unique} unique ids')
print(f'  duplicate groups differing on updated_at: {pairs_differ_updated_at} of {n_txn_dup_ids}')

# --- branches PK duplicates ---
n_branch_dup = int(branches['branch_id'].duplicated(keep=False).sum())
print(f'branches: {n_branch_dup} duplicate rows')

metrics['txn_dup_ids'] = n_txn_dup_ids
metrics['txn_unique'] = n_txn_unique
metrics['branch_dup_rows'] = n_branch_dup

## Section 3: Null and invalid values

Expected (transactions): amount null 179, amount negative 84 (min -98,442, 27 of them with status SUCCESS), promo_code null 4,838 (expected sparse, not an error). fraud_flag arrives as 'Y'/'N' strings.

In [ ]:
# amount: blanks -> NaN, then numeric.
amount = pd.to_numeric(transactions['amount'].replace('', np.nan), errors='coerce')
amount_null = int(amount.isna().sum())
amount_negative = int((amount < 0).sum())
amount_min = float(amount.min())
amount_zeros = int((amount == 0).sum())

status_upper = transactions['transaction_status'].str.upper()
neg_success = int(((amount < 0) & (status_upper == 'SUCCESS')).sum())

promo_null = int(transactions['promo_code'].replace('', np.nan).isna().sum())

print(f'amount null: {amount_null}')
print(f'amount negative: {amount_negative} (min {amount_min:,.0f}, zeros {amount_zeros})')
print(f'negative AND status=SUCCESS (logical contradiction): {neg_success}')
print(f'promo_code null (expected sparse, nullable): {promo_null}')
print('\nfraud_flag value counts:')
print(transactions['fraud_flag'].value_counts(dropna=False))

metrics['amount_null'] = amount_null
metrics['amount_negative'] = amount_negative
metrics['amount_min'] = amount_min
metrics['neg_success'] = neg_success
metrics['promo_null'] = promo_null

## Section 4: Broken foreign keys

Expected: `customer_id = 'C99999'` 80 rows (sentinel, not in customers), `branch_id = 'B999'` 80 rows (sentinel, not in branches). Both map to the unknown member (_sk = -1) downstream.

In [ ]:
cust_id_set = set(customers['customer_id'])
branch_id_set = set(branches['branch_id'])

fk_c99999 = int((transactions['customer_id'] == 'C99999').sum())
fk_b999 = int((transactions['branch_id'] == 'B999').sum())

txn_cust_unmatched = int((~transactions['customer_id'].isin(cust_id_set)).sum())
txn_branch_unmatched = int((~transactions['branch_id'].isin(branch_id_set)).sum())

print(f"customer_id = 'C99999': {fk_c99999} rows | in customers? {'C99999' in cust_id_set}")
print(f"branch_id = 'B999': {fk_b999} rows | in branches? {'B999' in branch_id_set}")
print(f'total transactions with unmatched customer_id: {txn_cust_unmatched}')
print(f'total transactions with unmatched branch_id: {txn_branch_unmatched}')

metrics['fk_c99999'] = fk_c99999
metrics['fk_b999'] = fk_b999

## Section 5: Currency distribution

Expected: USD 6,254 / IDR 6,046. There is no FX rate in source, so `SUM(amount)` across currencies is invalid without normalization; `dim_exchange_rate` supplies `rate_to_idr`.

In [ ]:
currency = transactions['currency'].str.upper()
currency_counts = currency.value_counts(dropna=False)
print(currency_counts)

ax = currency_counts.plot(kind='bar', color=['#1f77b4', '#ff7f0e'], figsize=(5, 3))
ax.set_title('Transaction count by currency')
ax.set_xlabel('currency')
ax.set_ylabel('rows')
plt.tight_layout()
plt.show()

metrics['currency_usd'] = int(currency_counts.get('USD', 0))
metrics['currency_idr'] = int(currency_counts.get('IDR', 0))

## Section 6: Late-arrival lag

Lag = `updated_at - transaction_date`, in days. Drives the incremental lookback window.

Expected: range 0.04 to 2.0 days, mean ~1.02, 6,179 rows with lag > 1 day, max 2 days. A 3-day lookback safely covers the 2-day maximum.

In [ ]:
# Source mixes datetime and date strings, so parse with mixed format and coerce.
txn_dt = pd.to_datetime(transactions['transaction_date'], errors='coerce', format='mixed')
upd_dt = pd.to_datetime(transactions['updated_at'], errors='coerce', format='mixed')

lag_days = (upd_dt - txn_dt).dt.total_seconds() / 86400.0
lag_valid = lag_days.dropna()

lag_mean = float(lag_valid.mean())
lag_min = float(lag_valid.min())
lag_max = float(lag_valid.max())
lag_gt_1day = int((lag_valid > 1.0).sum())

print(f'lag days: min {lag_min:.2f}, mean {lag_mean:.2f}, max {lag_max:.2f}')
print(f'rows with lag > 1 day: {lag_gt_1day}')
print(f'recommended lookback: >= 3 days (covers {lag_max:.2f}-day max)')

plt.figure(figsize=(7, 3))
plt.hist(lag_valid, bins=40, color='#2ca02c', edgecolor='white')
plt.axvline(1.0, color='red', linestyle='--', label='1 day')
plt.axvline(lag_mean, color='black', linestyle=':', label=f'mean {lag_mean:.2f}')
plt.title('Late-arrival lag distribution (updated_at - transaction_date)')
plt.xlabel('lag (days)')
plt.ylabel('rows')
plt.legend()
plt.tight_layout()
plt.show()

metrics['lag_mean'] = round(lag_mean, 2)
metrics['lag_max'] = round(lag_max, 2)
metrics['lag_gt_1day'] = lag_gt_1day

## Section 7: Future-dated records

Records dated after the reference date. Phase 1 (measured 2026-06-03): transaction_date 64, updated_at 87; max transaction_date 2026-06-05, max updated_at 2026-06-07.

These counts are date-relative and drift as the run date advances. Set `REFERENCE_DATE = '2026-06-03'` in the parameter cell to reproduce the Phase 1 figures exactly.

In [ ]:
ref = pd.Timestamp(REFERENCE_DATE) if REFERENCE_DATE else pd.Timestamp.today().normalize()

future_txn_date = int((txn_dt.dt.normalize() > ref).sum())
future_updated_at = int((upd_dt.dt.normalize() > ref).sum())

print(f'reference date: {ref.date()}')
print(f'transaction_date > reference: {future_txn_date} rows')
print(f'updated_at > reference: {future_updated_at} rows')
print(f'max transaction_date: {txn_dt.max()}')
print(f'max updated_at: {upd_dt.max()}')

metrics['future_txn_date'] = future_txn_date
metrics['future_updated_at'] = future_updated_at

## Section 8: Inactive status

Inactive records are kept (not filtered at ingestion); status is carried to the marts so reporting can filter intentionally.

Expected: customers INACTIVE 3,556 of 7,100 (50.1%); branches INACTIVE 13 of 30 (43.3%). 6,196 transactions reference an inactive customer; 5,333 reference an inactive branch.

In [ ]:
inactive_cust = int((customers['customer_status'].str.upper() == 'INACTIVE').sum())
inactive_branch = int((branches['branch_status'].str.upper() == 'INACTIVE').sum())
print(f'customers INACTIVE: {inactive_cust} of {len(customers)} ({inactive_cust/len(customers):.1%})')
print(f'branches INACTIVE: {inactive_branch} of {len(branches)} ({inactive_branch/len(branches):.1%})')

# customer_status / branch_status never conflict within duplicate pairs, so a
# de-duplicated id -> status lookup is unambiguous.
cust_status_map = (customers.drop_duplicates('customer_id')
                            .set_index('customer_id')['customer_status'].str.upper())
branch_status_map = (branches.drop_duplicates('branch_id')
                             .set_index('branch_id')['branch_status'].str.upper())

txn_ref_inactive_cust = int(transactions['customer_id'].map(cust_status_map).eq('INACTIVE').sum())
txn_ref_inactive_branch = int(transactions['branch_id'].map(branch_status_map).eq('INACTIVE').sum())
print(f'transactions referencing an inactive customer: {txn_ref_inactive_cust}')
print(f'transactions referencing an inactive branch: {txn_ref_inactive_branch}')

metrics['inactive_cust'] = inactive_cust
metrics['inactive_branch'] = inactive_branch
metrics['txn_ref_inactive_cust'] = txn_ref_inactive_cust
metrics['txn_ref_inactive_branch'] = txn_ref_inactive_branch

## Section 9: Branch name vs city inconsistency

Expected: 27 of 30 branches have a `branch_name` whose embedded location does not match the `city` column. Both fields are carried downstream; the data is not auto-corrected.

Rule used here: a row is consistent only when the `city` value appears as a substring (case-insensitive) inside `branch_name`.

In [ ]:
def name_contains_city(row):
    name = str(row['branch_name']).upper()
    city = str(row['city']).upper().strip()
    return bool(city) and (city in name)

consistent = branches.apply(name_contains_city, axis=1)
branch_mismatch = int((~consistent).sum())
print(f'branch_name vs city mismatch: {branch_mismatch} of {len(branches)}')
display(branches.loc[~consistent, ['branch_id', 'branch_name', 'city']].head(30))

metrics['branch_name_city_mismatch'] = branch_mismatch

## Summary: computed vs Phase 1

Every metric measured above, side by side with the Phase 1 reference value. The `match` column flags any drift. Future-dated counts are date-relative and excluded from the match check (they depend on `REFERENCE_DATE`).

In [ ]:
expected = {
    'customers_rows': 7100, 'transactions_rows': 12300, 'branches_rows': 30,
    'cust_dup_ids': 100, 'cust_dup_rows': 200,
    'conflict_email': 100, 'conflict_phone': 100, 'conflict_birth_date': 100,
    'conflict_occupation': 82, 'conflict_income_band': 68, 'conflict_kyc_status': 66,
    'txn_dup_ids': 300, 'txn_unique': 12000, 'branch_dup_rows': 0,
    'amount_null': 179, 'amount_negative': 84, 'amount_min': -98442.0, 'neg_success': 27,
    'promo_null': 4838,
    'fk_c99999': 80, 'fk_b999': 80,
    'currency_usd': 6254, 'currency_idr': 6046,
    'lag_mean': 1.02, 'lag_max': 2.0, 'lag_gt_1day': 6179,
    'inactive_cust': 3556, 'inactive_branch': 13,
    'txn_ref_inactive_cust': 6196, 'txn_ref_inactive_branch': 5333,
    'branch_name_city_mismatch': 27,
}

rows = []
for key, exp in expected.items():
    got = metrics.get(key)
    if isinstance(exp, float):
        match = got is not None and abs(float(got) - exp) < 0.05
    else:
        match = got == exp
    rows.append({'metric': key, 'computed': got, 'phase1_expected': exp, 'match': match})

summary = pd.DataFrame(rows)
display(summary)
print(f"\n{int(summary['match'].sum())} of {len(summary)} metrics match Phase 1.")
print('Future-dated (date-relative):',
      f"transaction_date={metrics.get('future_txn_date')}, updated_at={metrics.get('future_updated_at')}")